<a href="https://colab.research.google.com/github/Hanish-kumar/Plantinum_Rx/blob/main/platinumRx_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('platinum_rx_hotel.db')

# --- 1. Schema Setup ---
conn.execute("CREATE TABLE IF NOT EXISTS users (user_id INT, name TEXT)")
conn.execute("CREATE TABLE IF NOT EXISTS items (item_id INT, item_name TEXT, item_rate DECIMAL)")
conn.execute("CREATE TABLE IF NOT EXISTS bookings (booking_id INT, user_id INT, room_no INT, booking_date DATE)")
conn.execute("CREATE TABLE IF NOT EXISTS booking_commercials (bill_id INT, booking_id INT, item_id INT, item_quantity INT, bill_date DATE)")

# Insert Mock Data
conn.execute("INSERT OR REPLACE INTO users VALUES (1, 'Hanish'), (2, 'Kumar')")
conn.execute("INSERT OR REPLACE INTO items VALUES (1, 'Room Service', 500), (2, 'Laundry', 200)")
conn.execute("INSERT OR REPLACE INTO bookings VALUES (101, 1, 301, '2021-11-10'), (102, 1, 305, '2021-11-15'), (103, 2, 202, '2021-10-15')")
conn.execute("INSERT OR REPLACE INTO booking_commercials VALUES (501, 101, 1, 2, '2021-11-10'), (502, 102, 1, 1, '2021-11-15'), (503, 103, 1, 3, '2021-10-15')")
conn.commit()

# --- 2. Queries ---
# Q1: Last booked room for every user
q1 = "SELECT user_id, room_no FROM bookings GROUP BY user_id HAVING MAX(booking_date)"

# Q2: Total billing amount for November 2021
q2 = """
SELECT b.booking_id, SUM(bc.item_quantity * i.item_rate) as total_bill
FROM bookings b
JOIN booking_commercials bc ON b.booking_id = bc.booking_id
JOIN items i ON bc.item_id = i.item_id
WHERE b.booking_date BETWEEN '2021-11-01' AND '2021-11-30'
GROUP BY b.booking_id
"""

# Q3: Bills > 1000
q3 = """
SELECT bill_id, SUM(item_quantity * item_rate) as amount
FROM booking_commercials bc
JOIN items i ON bc.item_id = i.item_id
GROUP BY bill_id
HAVING amount > 1000
"""

print("Hotel Query 1 (Last Booked Room):")
display(pd.read_sql_query(q1, conn))

print("\nHotel Query 2 (Billing for November):")
display(pd.read_sql_query(q2, conn))

print("\nHotel Query 3 (Bills > 10)")
display(pd.read_sql_query(q3, conn))

# --- Q4: Most and Least Ordered Item per Month ---
# Logic: We count orders per month/item and rank them to find top (1) and bottom.
q4 = """
WITH MonthlyCounts AS (
    SELECT
        strftime('%m', bill_date) as month,
        item_id,
        SUM(item_quantity) as total_qty
    FROM booking_commercials
    GROUP BY 1, 2
),
RankedItems AS (
    SELECT
        month,
        item_id,
        total_qty,
        RANK() OVER(PARTITION BY month ORDER BY total_qty DESC) as rank_desc,
        RANK() OVER(PARTITION BY month ORDER BY total_qty ASC) as rank_asc
    FROM MonthlyCounts
)
SELECT month, item_id, total_qty,
       CASE WHEN rank_desc = 1 THEN 'Most Ordered' ELSE 'Least Ordered' END as status
FROM RankedItems
WHERE rank_desc = 1 OR rank_asc = 1;
"""

# --- Q5: 2nd Highest Bill Amount ---
# Logic: We calculate total for each bill, then find the one in the 2nd position.
q5 = """
SELECT bill_id, SUM(item_quantity * item_rate) as bill_amount
FROM booking_commercials bc
JOIN items i ON bc.item_id = i.item_id
GROUP BY bill_id
ORDER BY bill_amount DESC
LIMIT 1 OFFSET 1;
"""

print("\nHotel Query 4 (Most/Least Ordered):")
display(pd.read_sql_query(q4, conn))

print("\nHotel Query 5 (2nd Highest Bill):")
display(pd.read_sql_query(q5, conn))



Hotel Query 1 (Last Booked Room):


,user_id,room_no
0,1,305
1,2,202



Hotel Query 2 (Billing for November):


,booking_id,total_bill
0,101,729000
1,102,364500



Hotel Query 3 (Bills > 10)


,bill_id,amount
0,501,81000
1,502,40500
2,503,121500



Hotel Query 4 (Most/Least Ordered):


,month,item_id,total_qty,status
0,10,1,27,Most Ordered
1,11,1,27,Most Ordered



Hotel Query 5 (2nd Highest Bill):


,bill_id,bill_amount
0,501,81000


In [2]:
import sqlite3
import pandas as pd
import os

db_file = 'platinum_rx_clinic.db'
journal_file = f'{db_file}-journal'

# Attempt to remove database file and journal file if they exist
for f in [db_file, journal_file]:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f} to clear potential lock.")
        except OSError as e:
            print(f"Error removing {f}: {e}")

conn_clinic = sqlite3.connect(db_file)

try:
    # 1. Setup clinic_sales
    conn_clinic.execute("CREATE TABLE IF NOT EXISTS clinic_sales (uid INT, sales_channel TEXT, amount INT, datetime TIMESTAMP)")
    conn_clinic.execute("INSERT OR REPLACE INTO clinic_sales VALUES (1, 'App', 1500, '2021-05-10'), (2, 'Web', 2000, '2021-06-12')")
    conn_clinic.commit()

    # 2. Setup clinic_expenses
    conn_clinic.execute("CREATE TABLE IF NOT EXISTS clinic_expenses (expense_id INT, expense_amount INT, expense_date DATE)")
    conn_clinic.execute("INSERT OR REPLACE INTO clinic_expenses VALUES (501, 500, '2021-05-15'), (502, 800, '2021-06-20')")
    conn_clinic.commit()

    # Q1: Revenue per sales channel
    q_clinic_1 = "SELECT sales_channel, SUM(amount) as revenue FROM clinic_sales WHERE strftime('%Y', datetime) = '2021' GROUP BY sales_channel"

    print("Clinic Query 1 (Revenue by Channel):")
    display(pd.read_sql_query(q_clinic_1, conn_clinic))

    # Q3: Profit/Loss
    q_clinic_3 = """
    WITH MonthlyRevenue AS (
        SELECT
            strftime('%m', datetime) as month,
            SUM(amount) as total_revenue
        FROM clinic_sales
        WHERE strftime('%Y', datetime) = '2021'
        GROUP BY 1
    ),
    MonthlyExpenses AS (
        SELECT
            strftime('%m', expense_date) as month,
            SUM(expense_amount) as total_expenses
        FROM clinic_expenses
        WHERE strftime('%Y', expense_date) = '2021'
        GROUP BY 1
    )
    SELECT
        r.month,
        r.total_revenue,
        COALESCE(e.total_expenses, 0) as total_expenses,
        (r.total_revenue - COALESCE(e.total_expenses, 0)) as net_profit
    FROM MonthlyRevenue r
    LEFT JOIN MonthlyExpenses e ON r.month = e.month;
    """

    print("\nClinic Query 3 (Monthly Profit/Loss):")
    display(pd.read_sql_query(q_clinic_3, conn_clinic))

finally:
    conn_clinic.close()
    print("Database connection closed.")

Removed platinum_rx_clinic.db to clear potential lock.
Clinic Query 1 (Revenue by Channel):


,sales_channel,revenue
0,App,1500
1,Web,2000



Clinic Query 3 (Monthly Profit/Loss):


,month,total_revenue,total_expenses,net_profit
0,05,1500,500,1000
1,06,2000,800,1200


Database connection closed.


In [3]:
# 1. Setup Sample Dataframes
tickets = pd.DataFrame({
    'cms_id': ['C1', 'C2', 'C3'],
    'ticket_created_at': pd.to_datetime(['2023-01-01 10:00:00', '2023-01-01 12:30:00', '2023-01-01 15:00:00']),
    'resolved_at': pd.to_datetime(['2023-01-01 10:45:00', '2023-01-01 14:00:00', '2023-01-02 09:00:00']),
    'outlet_id': [101, 101, 102]
})
feedbacks = pd.DataFrame({'cms_id': ['C1', 'C2', 'C3']})

# 2. VLOOKUP Logic (Merging)
feedbacks_final = feedbacks.merge(tickets[['cms_id', 'ticket_created_at']], on='cms_id', how='left')

# 3. Time Analysis Logic (Same Day / Same Hour)
tickets['is_same_day'] = tickets['ticket_created_at'].dt.date == tickets['resolved_at'].dt.date
tickets['is_same_hour'] = (tickets['is_same_day']) & (tickets['ticket_created_at'].dt.hour == tickets['resolved_at'].dt.hour)

# 4. Count using Grouping (Pivot Table equivalent)
outlet_summary = tickets.groupby('outlet_id').agg(
    same_day_count=('is_same_day', 'sum'),
    same_hour_count=('is_same_hour', 'sum')
)
print("Spreadsheet Phase Output:")
display(outlet_summary)

Spreadsheet Phase Output:


,same_day_count,same_hour_count
outlet_id,,
101,2,1
102,0,0


In [4]:
def convert_minutes(total_minutes):
    hours = total_minutes // 60
    minutes = total_minutes % 60
    return f"{hours} hr{'s' if hours != 1 else ''} {minutes} minutes"

print(convert_minutes(130)) # Result: 2 hrs 10 minutes

2 hrs 10 minutes


In [5]:
def remove_duplicates(input_string):
    result = ""
    for char in input_string:
        if char not in result:
            result += char
    return result

print(remove_duplicates("platinuumrx")) # Result: platinmurx

platinumrx
